# 04 — Compare evaluation runs

Compare all `{run_id}/` folders under `data/paper_brief_evaluation/`. Show coverage, quality, and generator-token efficiency. Then list the worst-scored papers for one judged run.

**Prerequisite:** notebook 03 has written `03-evaluations.jsonl` for at least one run. This notebook reads files only. It does **not** call the generator or the judge. It does **not** query Postgres. It does **not** write files under `data/`. Run it with `just notebooks`. Do not use `just sandbox`.

Set **RUN_ID** to inspect an older judged run. Leave it empty to use the latest folder that already has `03-evaluations.jsonl` (lexicographic order is time order). Set **WORST_N** to how many lowest scores to list (default 10).

Do not import `paper_reviewer.flows`. Do not call `create_paper_brief` or `evaluate_paper_brief`.

In [1]:
# Folder name under data/paper_brief_evaluation/ (example "20260818T160000Z_llama3.1-8b").
# Leave empty to use the latest run that already has 03-evaluations.jsonl.
RUN_ID = ""

In [2]:
# How many lowest evaluation_score rows to list for the selected run.
WORST_N = 10

In [3]:
from __future__ import annotations

import json
import re
import statistics
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

from IPython.display import Markdown, display


def repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "paper_reviewer").is_dir() and (
            candidate / "pyproject.toml"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Cannot find the repo root. Start Jupyter with `just notebooks` "
        "so the kernel can see /workspace."
    )


REPO_ROOT = repo_root()
RUNS_PARENT = REPO_ROOT / "data" / "paper_brief_evaluation"
CORPUS_DIR = RUNS_PARENT / "corpus"
MANIFEST_PATH = CORPUS_DIR / "manifest.jsonl"
print(f"repo root: {REPO_ROOT}")
print(f"corpus dir: {CORPUS_DIR}")
print(f"runs parent: {RUNS_PARENT}")

repo root: /workspace
corpus dir: /workspace/data/paper_brief_evaluation/corpus
runs parent: /workspace/data/paper_brief_evaluation


In [4]:
_RUN_ID_PATTERN = re.compile(r"^\d{8}T\d{6}Z_.+$")
CRITERIA = (
    "faithfulness",
    "completeness",
    "conciseness",
    "topic_agnostic",
)
WEAK_MAX = 3.0
STRONG_MIN = 4.5
REASONING_LIMIT = 400


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            stripped = line.strip()
            if stripped:
                rows.append(json.loads(stripped))
    return rows


def has_brief(row: dict) -> bool:
    return row.get("brief") is not None


def as_usage_int(value: object) -> int | None:
    if isinstance(value, bool) or not isinstance(value, int):
        return None
    return value


def evaluation_score_of(row: dict) -> float | None:
    value = row.get("evaluation_score")
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return None
    return float(value)


def criterion_score_of(row: dict, name: str) -> int | None:
    evaluation = row.get("evaluation")
    if not isinstance(evaluation, dict):
        return None
    item = evaluation.get(name)
    if not isinstance(item, dict):
        return None
    score = item.get("score")
    if isinstance(score, bool) or not isinstance(score, int):
        return None
    return score


def criterion_reasoning_of(row: dict, name: str) -> str | None:
    evaluation = row.get("evaluation")
    if not isinstance(evaluation, dict):
        return None
    item = evaluation.get(name)
    if not isinstance(item, dict):
        return None
    reasoning = item.get("reasoning")
    if not isinstance(reasoning, str):
        return None
    return reasoning


def generator_slug(run_id: str) -> str:
    _stamp, sep, slug = run_id.partition("_")
    return slug if sep else ""


def list_run_dirs(parent: Path) -> list[Path]:
    dirs: list[Path] = []
    if not parent.is_dir():
        return dirs
    for child in parent.iterdir():
        if child.is_dir() and _RUN_ID_PATTERN.fullmatch(child.name):
            dirs.append(child)
    return sorted(dirs, key=lambda item: item.name)


def list_judged_run_ids(parent: Path) -> list[str]:
    ids: list[str] = []
    for run_dir in list_run_dirs(parent):
        if (run_dir / "03-evaluations.jsonl").is_file():
            ids.append(run_dir.name)
    return ids


def resolve_judged_run_dir(parent: Path, run_id: str) -> Path:
    chosen = run_id.strip()
    if chosen:
        run_dir = parent / chosen
        evaluations_path = run_dir / "03-evaluations.jsonl"
        if not evaluations_path.is_file():
            raise RuntimeError(
                f"Missing {evaluations_path}. Run notebook 03 (offline judge) "
                "first, or pick a judged run_id."
            )
        return run_dir
    ids = list_judged_run_ids(parent)
    if not ids:
        raise RuntimeError(
            f"No run folder with 03-evaluations.jsonl under {parent}. "
            "Run notebook 03 (offline judge) first."
        )
    return parent / ids[-1]


def parse_worst_n(value: object) -> int:
    if value is None:
        raise RuntimeError(
            "WORST_N must be a positive integer. No worst-paper list was shown."
        )
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.isdigit():
            n = int(stripped)
        else:
            raise RuntimeError(
                "WORST_N must be a positive integer. "
                "No worst-paper list was shown."
            )
    elif isinstance(value, bool) or not isinstance(value, int):
        raise RuntimeError(
            "WORST_N must be a positive integer. No worst-paper list was shown."
        )
    else:
        n = value
    if n < 1:
        raise RuntimeError(
            "WORST_N must be a positive integer. No worst-paper list was shown."
        )
    return n


def nearest_percentile(values: list[float], percentile: float) -> float:
    ordered = sorted(values)
    if not ordered:
        raise ValueError("nearest_percentile needs at least one value")
    rank = (percentile / 100.0) * (len(ordered) - 1)
    return ordered[int(round(rank))]


def mean_rounded(values: list[float], places: int) -> float | None:
    if not values:
        return None
    quant = Decimal(10) ** -places
    total = sum((Decimal(str(value)) for value in values), start=Decimal(0))
    mean = (total / Decimal(len(values))).quantize(
        quant, rounding=ROUND_HALF_UP
    )
    return float(mean)


def round_places(value: float, places: int) -> float:
    quant = Decimal(10) ** -places
    return float(Decimal(str(value)).quantize(quant, rounding=ROUND_HALF_UP))


def share_label(count: int, total: int) -> str:
    if total == 0:
        return ""
    pct = (Decimal(count) / Decimal(total) * Decimal(100)).quantize(
        Decimal("0.1"), rounding=ROUND_HALF_UP
    )
    return f"{count} ({pct}%)"


def show_table(headers: list[str], rows: list[list[object]]) -> None:
    def cell(value: object) -> str:
        if value is None:
            return ""
        return str(value).replace("|", "\\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(cell(value) for value in row) + " |")
    display(Markdown("\n".join(lines)))


def load_manifest_titles(path: Path) -> dict[str, str]:
    titles: dict[str, str] = {}
    if not path.is_file():
        return titles
    for row in load_jsonl(path):
        doi = row.get("doi")
        title = row.get("title")
        if isinstance(doi, str) and doi and isinstance(title, str):
            titles[doi] = title
    return titles


def load_run(path: Path) -> dict:
    briefs_path = path / "02-briefs.jsonl"
    evaluations_path = path / "03-evaluations.jsonl"
    summary_path = path / "03-token-summary.json"
    judge_path = path / "03-judge-model.txt"
    brief_rows = load_jsonl(briefs_path) if briefs_path.is_file() else []
    evaluation_rows = (
        load_jsonl(evaluations_path) if evaluations_path.is_file() else []
    )
    token_summary = None
    if summary_path.is_file():
        token_summary = json.loads(summary_path.read_text(encoding="utf-8"))
        if not isinstance(token_summary, dict):
            token_summary = None
    judge_model = None
    if judge_path.is_file():
        judge_model = judge_path.read_text(encoding="utf-8").strip() or None
    return {
        "run_id": path.name,
        "path": path,
        "generator_slug": generator_slug(path.name),
        "judged": evaluations_path.is_file(),
        "judge_model": judge_model,
        "brief_rows": brief_rows,
        "evaluation_rows": evaluation_rows,
        "token_summary": token_summary,
    }


def load_all_runs(parent: Path) -> list[dict]:
    return [load_run(path) for path in list_run_dirs(parent)]


def brief_success_count(run: dict) -> int:
    return sum(1 for row in run["brief_rows"] if has_brief(row))


def brief_error_count(run: dict) -> int:
    return sum(1 for row in run["brief_rows"] if not has_brief(row))


def scored_rows(run: dict) -> list[dict]:
    return [
        row
        for row in run["evaluation_rows"]
        if evaluation_score_of(row) is not None
    ]


def judge_error_count(run: dict) -> int:
    return sum(
        1
        for row in run["evaluation_rows"]
        if evaluation_score_of(row) is None
    )


def scores_by_doi(evaluation_rows: list[dict]) -> dict[str, float]:
    by_doi: dict[str, float] = {}
    for row in evaluation_rows:
        doi = row.get("doi")
        score = evaluation_score_of(row)
        if isinstance(doi, str) and doi and score is not None:
            by_doi[doi] = score
    return by_doi


def summary_path_value(run: dict, *keys: str) -> object:
    current: object = run.get("token_summary")
    for key in keys:
        if not isinstance(current, dict):
            return None
        current = current.get(key)
    return current


def median_total_tokens(run: dict) -> float | None:
    median = summary_path_value(run, "tokens", "total_tokens", "median")
    if isinstance(median, bool):
        return None
    if isinstance(median, (int, float)):
        return float(median)
    values: list[int] = []
    for row in run["brief_rows"]:
        parsed = as_usage_int(row.get("total_tokens"))
        if parsed is not None:
            values.append(parsed)
    if not values:
        return None
    return float(statistics.median(values))


def mean_score_per_1k(run: dict) -> float | None:
    value = summary_path_value(
        run, "quality", "mean_score_per_1k_total_tokens"
    )
    if isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    eval_by_doi = scores_by_doi(run["evaluation_rows"])
    per_1k: list[float] = []
    for row in run["brief_rows"]:
        if not has_brief(row):
            continue
        doi = row.get("doi")
        total = as_usage_int(row.get("total_tokens"))
        if not isinstance(doi, str) or total is None or total <= 0:
            continue
        score = eval_by_doi.get(doi)
        if score is None:
            continue
        per_1k.append((score / total) * 1000)
    return mean_rounded(per_1k, 4)


def joined_with_score(run: dict) -> int | None:
    value = summary_path_value(run, "coverage", "joined_with_score")
    if isinstance(value, bool) or not isinstance(value, int):
        return None
    return value


def quality_stats(run: dict) -> dict:
    rows = scored_rows(run)
    scores = [evaluation_score_of(row) for row in rows]
    scores = [score for score in scores if score is not None]
    mean_from_summary = summary_path_value(
        run, "quality", "mean_evaluation_score"
    )
    if isinstance(mean_from_summary, bool):
        mean_from_summary = None
    if not isinstance(mean_from_summary, (int, float)):
        mean_from_summary = None
    stats: dict = {
        "mean_evaluation_score": (
            float(mean_from_summary)
            if mean_from_summary is not None
            else mean_rounded(scores, 2)
        ),
        "median_evaluation_score": None,
        "min_evaluation_score": None,
        "p10_evaluation_score": None,
        "weak_le_3": "",
        "strong_ge_4_5": "",
    }
    for name in CRITERIA:
        stats[f"mean_{name}"] = None
    if not scores:
        return stats
    stats["median_evaluation_score"] = round_places(
        float(statistics.median(scores)), 2
    )
    stats["min_evaluation_score"] = round_places(min(scores), 2)
    stats["p10_evaluation_score"] = round_places(
        nearest_percentile(scores, 10), 2
    )
    weak = sum(1 for score in scores if score <= WEAK_MAX)
    strong = sum(1 for score in scores if score >= STRONG_MIN)
    stats["weak_le_3"] = share_label(weak, len(scores))
    stats["strong_ge_4_5"] = share_label(strong, len(scores))
    for name in CRITERIA:
        criterion_values = [
            float(score)
            for row in rows
            if (score := criterion_score_of(row, name)) is not None
        ]
        stats[f"mean_{name}"] = mean_rounded(criterion_values, 2)
    return stats


def total_tokens_by_doi(brief_rows: list[dict]) -> dict[str, int]:
    by_doi: dict[str, int] = {}
    for row in brief_rows:
        doi = row.get("doi")
        total = as_usage_int(row.get("total_tokens"))
        if isinstance(doi, str) and doi and total is not None:
            by_doi[doi] = total
    return by_doi


def lowest_criterion(row: dict) -> tuple[str, int, str] | None:
    best_name: str | None = None
    best_score: int | None = None
    for name in CRITERIA:
        score = criterion_score_of(row, name)
        if score is None:
            continue
        if best_score is None or score < best_score:
            best_name = name
            best_score = score
    if best_name is None or best_score is None:
        return None
    reasoning = criterion_reasoning_of(row, best_name) or ""
    return best_name, best_score, reasoning


def truncate_text(text: str, limit: int) -> str:
    stripped = " ".join(text.split())
    if len(stripped) <= limit:
        return stripped
    return stripped[: limit - 3].rstrip() + "..."

## Inventory

Every `{run_id}/` folder. `corpus/` is not a run. A judged run has `03-evaluations.jsonl`. Incomplete folders stay here with empty score counts.

In [5]:
RUNS = load_all_runs(RUNS_PARENT)
JUDGED = [run for run in RUNS if run["judged"]]
TITLES = load_manifest_titles(MANIFEST_PATH)
print(f"run folders: {len(RUNS)}")
print(f"judged: {len(JUDGED)}")
print(f"manifest titles: {len(TITLES)}")
if not RUNS:
    print(f"No run folders under {RUNS_PARENT}. Run notebook 02 first.")
else:
    show_table(
        [
            "run_id",
            "generator",
            "judged",
            "brief_rows",
            "brief_errors",
            "scored",
            "judge_errors",
            "token_summary",
        ],
        [
            [
                run["run_id"],
                run["generator_slug"],
                "yes" if run["judged"] else "no",
                brief_success_count(run),
                brief_error_count(run),
                len(scored_rows(run)) if run["judged"] else "",
                judge_error_count(run) if run["judged"] else "",
                "yes" if run["token_summary"] is not None else "no",
            ]
            for run in RUNS
        ],
    )

run folders: 2
judged: 2
manifest titles: 109


| run_id | generator | judged | brief_rows | brief_errors | scored | judge_errors | token_summary |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 20260818T150139Z_llama3.1-8b | llama3.1-8b | yes | 109 | 0 | 109 | 0 | yes |
| 20260818T221210Z_gemma4-e4b | gemma4-e4b | yes | 104 | 5 | 96 | 8 | yes |

## Global comparison

One row per judged run. Prefer `03-token-summary.json` for mean score, joined coverage, median `total_tokens`, and score per 1k tokens. Recompute criterion means and score tails from `03-evaluations.jsonl`.

In [6]:
if not JUDGED:
    print(
        "No judged runs (missing 03-evaluations.jsonl). "
        "Run notebook 03 first."
    )
else:
    coverage_rows: list[list[object]] = []
    quality_rows: list[list[object]] = []
    cost_rows: list[list[object]] = []
    for run in JUDGED:
        quality = quality_stats(run)
        coverage_rows.append(
            [
                run["run_id"],
                run["generator_slug"],
                run["judge_model"] or "",
                brief_success_count(run),
                len(scored_rows(run)),
                brief_error_count(run),
                judge_error_count(run),
                joined_with_score(run),
            ]
        )
        quality_rows.append(
            [
                run["run_id"],
                quality["mean_evaluation_score"],
                quality["median_evaluation_score"],
                quality["min_evaluation_score"],
                quality["p10_evaluation_score"],
                quality["weak_le_3"],
                quality["strong_ge_4_5"],
                quality["mean_faithfulness"],
                quality["mean_completeness"],
                quality["mean_conciseness"],
                quality["mean_topic_agnostic"],
            ]
        )
        cost_rows.append(
            [
                run["run_id"],
                median_total_tokens(run),
                mean_score_per_1k(run),
            ]
        )
    display(Markdown("**Identity and coverage**"))
    show_table(
        [
            "run_id",
            "generator",
            "judge_model",
            "brief_rows",
            "scored",
            "brief_errors",
            "judge_errors",
            "joined_with_score",
        ],
        coverage_rows,
    )
    display(Markdown("**Quality (1–5)**"))
    show_table(
        [
            "run_id",
            "mean_evaluation_score",
            "median_evaluation_score",
            "min",
            "p10",
            "weak_le_3",
            "strong_ge_4_5",
            "mean_faithfulness",
            "mean_completeness",
            "mean_conciseness",
            "mean_topic_agnostic",
        ],
        quality_rows,
    )
    display(Markdown("**Cost / efficiency (generator tokens)**"))
    show_table(
        [
            "run_id",
            "median_total_tokens",
            "mean_score_per_1k_total_tokens",
        ],
        cost_rows,
    )

**Identity and coverage**

| run_id | generator | judge_model | brief_rows | scored | brief_errors | judge_errors | joined_with_score |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 20260818T150139Z_llama3.1-8b | llama3.1-8b | llama3.1:8b | 109 | 109 | 0 | 0 | 109 |
| 20260818T221210Z_gemma4-e4b | gemma4-e4b | gemma4:e4b | 104 | 96 | 5 | 8 | 96 |

**Quality (1–5)**

| run_id | mean_evaluation_score | median_evaluation_score | min | p10 | weak_le_3 | strong_ge_4_5 | mean_faithfulness | mean_completeness | mean_conciseness | mean_topic_agnostic |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 20260818T150139Z_llama3.1-8b | 4.31 | 4.5 | 1.0 | 3.75 | 2 (1.8%) | 65 (59.6%) | 4.5 | 3.95 | 4.22 | 4.55 |
| 20260818T221210Z_gemma4-e4b | 4.83 | 5.0 | 4.0 | 4.5 | 0 (0.0%) | 88 (91.7%) | 4.59 | 4.99 | 4.75 | 4.98 |

**Cost / efficiency (generator tokens)**

| run_id | median_total_tokens | mean_score_per_1k_total_tokens |
| --- | --- | --- |
| 20260818T150139Z_llama3.1-8b | 3116.0 | 1.3996 |
| 20260818T221210Z_gemma4-e4b | 3316.0 | 1.4987 |

## Fair overlap

Pairwise mean `evaluation_score` on shared DOIs. Use this when runs used `LIMIT` or had fail-soft gaps, so raw means are not comparable.

In [7]:
if len(JUDGED) < 2:
    print("Need two judged runs to compare overlapping DOIs.")
else:
    overlap_rows: list[list[object]] = []
    for index, left in enumerate(JUDGED):
        left_scores = scores_by_doi(left["evaluation_rows"])
        for right in JUDGED[index + 1 :]:
            right_scores = scores_by_doi(right["evaluation_rows"])
            shared = sorted(set(left_scores) & set(right_scores))
            if not shared:
                continue
            overlap_rows.append(
                [
                    left["run_id"],
                    right["run_id"],
                    len(shared),
                    mean_rounded([left_scores[doi] for doi in shared], 2),
                    mean_rounded([right_scores[doi] for doi in shared], 2),
                ]
            )
    if not overlap_rows:
        print("Judged runs share no scored DOIs.")
    else:
        show_table(
            [
                "run_a",
                "run_b",
                "shared_dois",
                "mean_score_a",
                "mean_score_b",
            ],
            overlap_rows,
        )

| run_a | run_b | shared_dois | mean_score_a | mean_score_b |
| --- | --- | --- | --- | --- |
| 20260818T150139Z_llama3.1-8b | 20260818T221210Z_gemma4-e4b | 96 | 4.32 | 4.83 |

## Worst papers (selected run)

Lowest `evaluation_score` success rows for `RUN_ID`. Empty `RUN_ID` uses the latest judged run. Skip JSONL error lines. Join title from `corpus/manifest.jsonl` and `total_tokens` from `02-briefs.jsonl`.

In [8]:
worst_n = parse_worst_n(WORST_N)
selected_dir = resolve_judged_run_dir(RUNS_PARENT, RUN_ID)
selected = load_run(selected_dir)
selected_scored = scored_rows(selected)
selected_scored.sort(
    key=lambda row: (
        evaluation_score_of(row) or 0.0,
        str(row.get("doi") or ""),
    )
)
WORST_ROWS = selected_scored[:worst_n]
tokens_by_doi = total_tokens_by_doi(selected["brief_rows"])
print(f"selected run: {selected['run_id']}")
print(f"judge model: {selected['judge_model'] or '(missing 03-judge-model.txt)'}")
print(f"scored: {len(selected_scored)}")
print(f"showing: {len(WORST_ROWS)} (WORST_N={worst_n})")
if not WORST_ROWS:
    print("No success rows in 03-evaluations.jsonl for this run.")
else:
    show_table(
        [
            "rank",
            "evaluation_score",
            "faithfulness",
            "completeness",
            "conciseness",
            "topic_agnostic",
            "doi",
            "title",
            "total_tokens",
        ],
        [
            [
                rank,
                evaluation_score_of(row),
                criterion_score_of(row, "faithfulness"),
                criterion_score_of(row, "completeness"),
                criterion_score_of(row, "conciseness"),
                criterion_score_of(row, "topic_agnostic"),
                row.get("doi") or "",
                TITLES.get(str(row.get("doi") or ""), ""),
                tokens_by_doi.get(str(row.get("doi") or "")),
            ]
            for rank, row in enumerate(WORST_ROWS, start=1)
        ],
    )

selected run: 20260818T221210Z_gemma4-e4b
judge model: gemma4:e4b
scored: 96
showing: 10 (WORST_N=10)


| rank | evaluation_score | faithfulness | completeness | conciseness | topic_agnostic | doi | title | total_tokens |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 4.0 | 2 | 5 | 4 | 5 | 10.1093/JME/TJAG095 | Effectiveness of natural predators, microbial bioinsecticides, and plant-based bioinsecticides to reduce mosquito-borne disease burden: a systematic review. | 2999 |
| 2 | 4.0 | 2 | 5 | 4 | 5 | 10.3390/VACCINES14060499 | Advances and Challenges in Vaccine Development for West Nile Virus (WNV) Infection. | 3214 |
| 3 | 4.0 | 2 | 5 | 4 | 5 | 10.64898/2026.05.10.722846 | Ecotypes, Wolbachia, and urbanization shape Culex pipiens population structure in a West Nile virus hotspot. | 3450 |
| 4 | 4.25 | 3 | 5 | 4 | 5 | 10.1016/J.APSB.2026.02.021 | Discovery and mutasynthetic optimization of ansatrienin B: A new broad-spectrum inhibitor against RNA viruses. | 3388 |
| 5 | 4.25 | 5 | 4 | 5 | 3 | 10.1038/S41467-026-73251-5 | Fine-scale heterogeneity and local amplification of West Nile virus in urban environments in Berlin. | 3051 |
| 6 | 4.25 | 2 | 5 | 5 | 5 | 10.1371/JOURNAL.PNTD.0014400 | Correction: Association of avian biodiversity and West Nile Virus circulation in Culex mosquitoes in Emilia-Romagna, Italy. | 2015 |
| 7 | 4.25 | 3 | 5 | 4 | 5 | 10.1371/JOURNAL.PONE.0342150 | Catch basin larvicide treatments impact adult mosquito West Nile virus vector species in metropolitan Milwaukee, WI, U.S.A. | 3429 |
| 8 | 4.25 | 3 | 5 | 4 | 5 | 10.2903/J.EFSA.2026.10231 | Surveillance of West Nile virus infections in humans and animals in Europe, monthly report - data submitted up to 24 June 2026. | 2123 |
| 9 | 4.5 | 3 | 5 | 5 | 5 | 10.1016/J.ONEHLT.2025.101310 | Genomic epidemiology and phylogeographic reconstruction of West Nile virus 2 in Italy from 2011 to 2023. | 3694 |
| 10 | 4.5 | 4 | 5 | 4 | 5 | 10.1021/ACSOMEGA.5C13455 | Protein Large Language Models Can Predict Flavivirus Protease Target Specificity. | 3400 |

## Lowest criterion (same rows)

For each worst paper, show the lowest G-Eval criterion and a truncated `reasoning`. Ties keep the first criterion in template order: faithfulness, completeness, conciseness, topic_agnostic.

In [9]:
if not WORST_ROWS:
    print("No worst-paper rows. Run the previous cell first.")
else:
    reasoning_rows: list[list[object]] = []
    for rank, row in enumerate(WORST_ROWS, start=1):
        lowest = lowest_criterion(row)
        if lowest is None:
            reasoning_rows.append(
                [rank, row.get("doi") or "", "", "", ""]
            )
            continue
        name, score, reasoning = lowest
        reasoning_rows.append(
            [
                rank,
                row.get("doi") or "",
                name,
                score,
                truncate_text(reasoning, REASONING_LIMIT),
            ]
        )
    show_table(
        ["rank", "doi", "lowest_criterion", "score", "reasoning"],
        reasoning_rows,
    )

| rank | doi | lowest_criterion | score | reasoning |
| --- | --- | --- | --- | --- |
| 1 | 10.1093/JME/TJAG095 | faithfulness | 2 | The brief claims that 'Fifteen out of eighteen included field studies demonstrated a reduction in disease burden within the intervention groups.' The full text states: 'Only five studies performed statistical analysis to compare intervention and control groups, two of which found no significant association between the intervention and the disease outcome. The remaining sixteen reduced disease b... |
| 2 | 10.3390/VACCINES14060499 | faithfulness | 2 | The brief claims that 'the coronavirus genome encodes numerous proteins (C, prM/M, E, NS1 through NS5)'. The article text discusses WNV structure and lists structural proteins (capsid (C), premembrane/membrane (prM/M) and envelope (E)) and non-structural proteins (NS1, NS2A, NS2B, NS3, NS4A, NS4B, and NS5). However, it never mentions 'coronavirus' or the specific protein list structure provided... |
| 3 | 10.64898/2026.05.10.722846 | faithfulness | 2 | The brief claims that 'Three key genomic regions on chromosomes 1 and 3 displayed high LD, low divergence from background, and characteristic patterns consistent with polymorphic chromosomal inversions.' The text confirms the identification of substantial genetic differentiation in three regions on chromosomes 1 and 3 showing high linkage disequilibrium (LD) and patterns consistent with chromos... |
| 4 | 10.1016/J.APSB.2026.02.021 | faithfulness | 3 | The brief claims that ansatrienin B functions as a new broad-spectrum inhibitor against RNA viruses, including promising activity against SARS-CoV-2. The text confirms testing against SARS-CoV-2 (Section 2.3 and 2.7). It also mentions evaluating inhibition across varying concentrations to derive IC50 values (Section 2.3), which supports the 'inhibitor' claim for SARS-CoV-2. However, the full te... |
| 5 | 10.1038/S41467-026-73251-5 | topic_agnostic | 3 | The content is specialized (entomology/virology surveillance), making it highly specific. However, since no question was asked, I cannot assess its utility without context. |
| 6 | 10.1371/JOURNAL.PNTD.0014400 | faithfulness | 2 | The brief claims that the text 'implies a refinement of previous findings regarding the association between avian biodiversity and mosquito-borne viral threats in Emilia-Romagna, Italy' (summary). The full text is an 'Erratum' correcting author affiliations and providing citation information for a previously published article. It does not contain any discussion or implication about refining fin... |
| 7 | 10.1371/JOURNAL.PONE.0342150 | faithfulness | 3 | The brief claims that 'Applying microbial larvicides to catch basins significantly reduces the abundance of West Nile virus vector mosquitoes, as demonstrated by reduced larval and pupal presence in treated areas compared to untreated controls.' The full text confirms the study evaluated larval/pupal abundance using *L. sphaericus* treatments (Section: Catch basin larvicide applications and eva... |
| 8 | 10.2903/J.EFSA.2026.10231 | faithfulness | 3 | The brief claims that five WNV animal outbreaks were reported in Europe: one among equids (in France) and four among birds (in Italy). The full text states, "From the veterinary perspective, five WNV outbreaks have been reported in Europe in 2026: one among equids and four among birds." It also specifies that the equid outbreak was reported by France. However, the brief's key finding regarding... |
| 9 | 10.1016/J.ONEHLT.2025.101310 | faithfulness | 3 | The brief claims the analysis was conducted in Italy between 2011 and 2023. The text states that a dataset of 334 whole genome sequences included European WNV-2 complete genome sequences available on public databases with collection dates from 2004 to 2023, and specifically notes the Italian subset included 226 sequences. While it mentions Italy's inclusion, the specific date range (2011-2023)... |
| 10 | 10.1021/ACSOMEGA.5C13455 | faithfulness | 4 | The brief correctly states that analyzing cleavage sites from multiple flaviviruses reveals broad target specificity, challenging traditional motifs. It also mentions the PICS analysis showed variability across subsite positions (P2 through P5'). The finding about LLMs being tested against positive samples matched to negative decoys is supported by the text. However, the brief overstates the co... |